# Bag-of-Words SL

Sample script to exemplify **instantiating** and **supervised-learning** on the bag-of-words dataset. 

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.sl import BagOfWordsSLConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
    mse_expr
)

repo_root = get_repo_base()
device = torch.device("cuda:0")

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/sl.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Configure

In [2]:
config = BagOfWordsSLConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-sl-example",
    corr=0.2,
    aux_words_ratio=0.5,
    train_epochs=2,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset corr target: {config.data.corr:.4f}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

Study folder: /home/nlyu/Code/maxrl-statistics/artifacts/bow-sl-example/7-words_corr-0.2_len-128_pow-1.0_ar-0.5
Dataset corr target: 0.2000
Backbone lr: 1.230e-03
Head lr:     8.192e-03


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [3]:
state.run_training()

sl epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

sl epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

## Results

Training saves: 
1. A compact `metrics.parquet` to disk which contains per-epoch sufficient statistics to compute metrics. 
2. Validation parquet containing per-row ground-truth and target

In [4]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,train_ground_truth_xx,train_ground_truth_xy,train_ground_truth_yy,train_ground_truth_n,val_target_xx,val_target_xy,val_target_yy,val_target_n,val_ground_truth_xx,val_ground_truth_xy,val_ground_truth_yy,val_ground_truth_n
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,8391.862305,1260.597046,49548.195312,49984.0,8391.862305,1224.186523,1973.484009,49984.0,8484.947266,1891.105469,49494.257812,49920.0,8484.947266,1856.835815,1999.744507,49920.0
1,4244.665527,1800.941772,49542.074219,49984.0,4244.665527,1641.667847,1973.260986,49984.0,2107.140137,1715.529541,49494.257812,49920.0,2107.140137,1723.812622,1999.744507,49920.0


In [ ]:
analysis = BagOfWordsAnalysisConfig.from_grouped({"example": [(0, config.study_folder)]})

analysis.plot_vs_epoch([
    rsq_expr(split="train", y="ground_truth"),
    rsq_expr(split="val", y="ground_truth"),
    corr_expr(split="train", y="ground_truth"),
    corr_expr(split="val", y="ground_truth"),
])


In [7]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()

model_preds,ground_truth,target
f64,f64,f64
0.182617,0.015564,0.761719
-0.292969,-0.263672,-0.055908
-0.061523,0.0,-0.238281
0.245117,0.046631,-0.679688
-0.337891,-0.279297,0.376953
